# Shifu OCR — Colab Training
Train character landscapes from 50k real medical text images.
Upload your training data, run training with multiprocessing, download the model.

In [ ]:
# Step 1: Install dependencies
!pip install numpy Pillow scipy scikit-image -q

In [ ]:
# Step 2: Upload training data
# Option A: Upload zip from your machine
from google.colab import files
import os, zipfile

print('Upload your training_data.zip (contains images/ + train_list.txt + val_list.txt)')
print('Or skip this cell if using Google Drive')
uploaded = files.upload()

for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name, 'r') as z:
            z.extractall('training_data')
        print(f'Extracted {name}')

TRAINING_DIR = 'training_data'
if os.path.exists('training_data/training_data'):
    TRAINING_DIR = 'training_data/training_data'

# Check
img_dir = os.path.join(TRAINING_DIR, 'images')
n_images = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
print(f'Found {n_images} training images in {TRAINING_DIR}')

In [ ]:
# Step 2b (Alternative): Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# TRAINING_DIR = '/content/drive/MyDrive/shifu_training_data'  # adjust path

In [ ]:
# Step 3: Shifu OCR Engine (self-contained)
import numpy as np
import json
import time
from PIL import Image
from scipy import ndimage
from skimage import morphology, filters
from collections import defaultdict
import multiprocessing as mp
from functools import partial
import warnings
warnings.filterwarnings('ignore')

# ── Feature extraction ──────────────────────────────────────

def fast_binarize(grayscale_image):
    from skimage.filters import threshold_otsu
    try:
        thresh = threshold_otsu(grayscale_image)
    except ValueError:
        thresh = 128
    binary = (grayscale_image < thresh).astype(np.uint8)
    return morphology.remove_small_objects(binary.astype(bool), min_size=5).astype(np.uint8)

def extract_region(binary, pad=3):
    coords = np.argwhere(binary > 0)
    if len(coords) == 0:
        return binary
    r0, c0 = np.maximum(coords.min(axis=0) - pad, 0)
    r1, c1 = np.minimum(coords.max(axis=0) + pad, np.array(binary.shape) - 1)
    return binary[r0:r1+1, c0:c1+1]

def normalize_region(region, size=(64, 64)):
    img = Image.fromarray((region * 255).astype(np.uint8)).resize(size, Image.NEAREST)
    return (np.array(img) > 127).astype(np.uint8)

def extract_features(br):
    h, w = br.shape
    feats = []
    # Topology
    padded = np.pad(br, 1, mode='constant', constant_values=0)
    _, nfg = ndimage.label(padded)
    _, nbg = ndimage.label(1 - padded)
    holes = nbg - 1
    feats.extend([float(nfg), float(holes), float(nfg - holes)])
    feats.append(float(np.mean(br)))
    feats.append(float(np.mean(br == np.fliplr(br))) if w >= 2 else 1.0)
    feats.append(float(np.mean(br == np.flipud(br))) if h >= 2 else 1.0)
    total = br.sum()
    if total > 0 and h > 0 and w > 0:
        rows, cols = np.arange(h).reshape(-1, 1), np.arange(w).reshape(1, -1)
        feats.append(float((br * rows).sum() / (total * h)))
        feats.append(float((br * cols).sum() / (total * w)))
    else:
        feats.extend([0.5, 0.5])
    # Quadrants
    mh, mw = h // 2, w // 2
    quads = [
        float(br[:mh, :mw].mean()) if mh > 0 and mw > 0 else 0,
        float(br[:mh, mw:].mean()) if mh > 0 else 0,
        float(br[mh:, :mw].mean()) if mw > 0 else 0,
        float(br[mh:, mw:].mean()),
    ]
    qt = sum(quads)
    feats.extend([q / qt if qt > 0 else 0.25 for q in quads])
    # Projections
    for axis in [1, 0]:
        raw = br.mean(axis=axis)
        proj = np.zeros(6)
        bw = max(1, len(raw) // 6)
        for i in range(6):
            s, e = i * bw, min((i + 1) * bw, len(raw))
            if s < len(raw): proj[i] = raw[s:e].mean()
        pt = proj.sum()
        if pt > 0: proj /= pt
        feats.extend(proj.tolist())
    # Crossings
    for axis in [0, 1]:
        for i in range(6):
            if axis == 0:
                row = min(int((i + 0.5) * h / 6), h - 1)
                line = br[row, :]
            else:
                col = min(int((i + 0.5) * w / 6), w - 1)
                line = br[:, col]
            feats.append(float(np.abs(np.diff(line.astype(int))).sum() / 2))
    # Endpoints/junctions
    if h >= 4 and w >= 4 and br.sum() >= 10:
        try:
            skel = morphology.skeletonize(br.astype(bool))
            nc = ndimage.convolve(skel.astype(int), np.ones((3, 3), dtype=int), mode='constant') - skel.astype(int)
            _, n_ep = ndimage.label(skel & (nc == 1))
            _, n_jp = ndimage.label(skel & (nc >= 3))
            feats.extend([float(n_ep), float(n_jp)])
        except:
            feats.extend([0.0, 0.0])
    else:
        feats.extend([0.0, 0.0])

    # ── v2 geometric features ──
    ink = br > 0
    ink_pixels = ink.sum()
    ink_coords = np.argwhere(ink)
    if len(ink_coords) >= 2:
        ink_h = ink_coords[:, 0].max() - ink_coords[:, 0].min() + 1
        ink_w = ink_coords[:, 1].max() - ink_coords[:, 1].min() + 1
        feats.append(float(ink_h / max(ink_w, 1)))
    else:
        feats.append(1.0)
    feats.append(float(ink_pixels / max(h * w, 1)))
    t3 = h // 3
    feats.append(float(br[:t3, :].mean()) if t3 > 0 else 0)
    feats.append(float(br[t3:2*t3, :].mean()) if t3 > 0 else 0)
    feats.append(float(br[2*t3:, :].mean()) if t3 > 0 else 0)
    # Stroke width (vectorized)
    diffs = np.diff(np.pad(br, ((0, 0), (1, 1)), constant_values=0), axis=1)
    starts = np.where(diffs == 1)
    ends = np.where(diffs == -1)
    if len(starts[0]) > 0 and len(starts[0]) == len(ends[0]):
        run_lengths = ends[1] - starts[1]
        feats.append(float(run_lengths.mean() / max(w, 1)))
        feats.append(float(run_lengths.std() / max(w, 1)))
    else:
        feats.extend([0.0, 0.0])
    # Top disconnection
    v_proj = br.mean(axis=1)
    gap_rows = (v_proj < 0.01).astype(int)
    top_half_gaps = gap_rows[:h // 2]
    max_gap = 0
    current_gap = 0
    for g in top_half_gaps:
        if g: current_gap += 1
        else: max_gap = max(max_gap, current_gap); current_gap = 0
    max_gap = max(max_gap, current_gap)
    feats.append(float(max_gap / max(h // 2, 1)))
    # Mid horizontal extent
    mid_slice = br[t3:2*t3, :] if t3 > 0 else br
    mid_col_ink = (mid_slice.sum(axis=0) > 0).sum()
    feats.append(float(mid_col_ink / max(w, 1)))
    # Ascender/descender
    if ink_pixels > 0:
        feats.append(float(br[:t3, :].sum() / ink_pixels))
        feats.append(float(br[2*t3:, :].sum() / ink_pixels))
    else:
        feats.extend([0.33, 0.33])
    # Edge straightness
    row_has_ink = br.any(axis=1)
    if row_has_ink.sum() >= 3:
        ink_rows = np.where(row_has_ink)[0]
        left_edge = np.array([np.where(br[r, :] > 0)[0][0] for r in ink_rows])
        right_edge = np.array([np.where(br[r, :] > 0)[0][-1] for r in ink_rows])
        feats.append(1.0 - float(np.std(left_edge) / max(w, 1)))
        feats.append(1.0 - float(np.std(right_edge) / max(w, 1)))
    else:
        feats.extend([0.5, 0.5])
    # Openness
    feats.append(1.0 - float(br[0, :].mean()) if h > 0 else 0)
    feats.append(1.0 - float(br[-1, :].mean()) if h > 0 else 0)

    return np.array(feats, dtype=float)

print(f'Feature vector length: {len(extract_features(np.zeros((64,64), dtype=np.uint8)))}')

In [ ]:
# Step 4: Landscape class with Welford online stats

class Landscape:
    def __init__(self, label):
        self.label = label
        self.mean = None
        self.variance = None
        self._m2 = None
        self.n = 0
        self.n_correct = 0
        self.n_errors = 0
        self.confused_with = defaultdict(int)

    def absorb(self, fv):
        self.n += 1
        if self.n == 1:
            self.mean = fv.copy()
            self._m2 = np.zeros_like(fv)
            self.variance = np.ones_like(fv) * 2.0
        else:
            delta = fv - self.mean
            self.mean += delta / self.n
            delta2 = fv - self.mean
            self._m2 += delta * delta2
            raw_var = self._m2 / self.n
            self.variance = np.maximum(raw_var, 0.1 / np.sqrt(self.n))

    def fit(self, fv):
        if self.mean is None: return -float('inf')
        diff = fv - self.mean
        precision = 1.0 / (self.variance + 1e-8)
        return -0.5 * np.sum(diff ** 2 * precision) + np.log(self.n + 1) * 0.5

    def to_dict(self):
        return {
            'label': self.label, 'n': self.n,
            'mean': self.mean.tolist() if self.mean is not None else None,
            'variance': self.variance.tolist() if self.variance is not None else None,
            'n_correct': self.n_correct, 'n_errors': self.n_errors,
            'confused_with': dict(self.confused_with),
        }

print('Landscape class ready')

In [ ]:
# Step 5: Parallel feature extraction

ALLOWED_CHARS = set('abcdefghijklmnopqrstuvwxyz0123456789')

def segment_characters(grayscale_image, min_char_width=3):
    from skimage.filters import threshold_otsu
    try: thresh = threshold_otsu(grayscale_image)
    except: thresh = 128
    binary = (grayscale_image < thresh).astype(np.uint8)
    binary = morphology.remove_small_objects(binary.astype(bool), min_size=5).astype(np.uint8)
    v_proj = binary.sum(axis=0).astype(float)
    is_ink = v_proj > 0
    segments = []
    in_char = False
    start = 0
    for i in range(len(is_ink)):
        if is_ink[i] and not in_char: start = i; in_char = True
        elif not is_ink[i] and in_char:
            if i - start >= min_char_width: segments.append((start, i))
            in_char = False
    if in_char and len(is_ink) - start >= min_char_width:
        segments.append((start, len(is_ink)))
    char_images = []
    for c_start, c_end in segments:
        col_slice = binary[:, c_start:c_end]
        row_proj = col_slice.sum(axis=1)
        rows_with_ink = np.where(row_proj > 0)[0]
        if len(rows_with_ink) == 0: continue
        r_start = max(0, rows_with_ink[0] - 2)
        r_end = min(binary.shape[0], rows_with_ink[-1] + 3)
        char_crop = grayscale_image[r_start:r_end, c_start:c_end]
        ch, cw = char_crop.shape
        size = max(ch, cw) + 10
        padded = np.full((size, size), 255, dtype=np.uint8)
        y_off = (size - ch) // 2
        x_off = (size - cw) // 2
        padded[y_off:y_off+ch, x_off:x_off+cw] = char_crop
        char_images.append(padded)
    return char_images

def process_image(args):
    """Process one image: segment, align, extract features. Returns list of (label, features)."""
    img_path, label = args
    try:
        img = Image.open(img_path)
        if img.mode != 'L': img = img.convert('L')
        arr = np.array(img)
        img.close()
        segments = segment_characters(arr, min_char_width=2)
        if not segments: return []
        # Align to label
        clean_chars = [c.lower() for c in label if c.lower() in ALLOWED_CHARS]
        if not clean_chars: return []
        ratio = len(segments) / max(len(clean_chars), 1)
        if ratio < 0.8 or ratio > 1.2: return []
        n = min(len(segments), len(clean_chars))
        results = []
        for i in range(n):
            binary = fast_binarize(segments[i])
            region = normalize_region(extract_region(binary))
            fv = extract_features(region)
            results.append((clean_chars[i], fv))
        return results
    except:
        return []

print('Parallel processor ready')

In [ ]:
# Step 6: TRAIN with multiprocessing

import os, time

# Load training list
train_list_path = os.path.join(TRAINING_DIR, 'train_list.txt')
entries = []
with open(train_list_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line or '\t' not in line: continue
        img_path, label = line.split('\t', 1)
        full_path = os.path.join(TRAINING_DIR, img_path)
        if os.path.exists(full_path):
            entries.append((full_path, label))

print(f'Loaded {len(entries)} training images')

# Process in parallel
n_cores = mp.cpu_count()
print(f'Using {n_cores} CPU cores')

start = time.time()
landscapes = {}
total_chars = 0
batch_size = 2000

for batch_start in range(0, len(entries), batch_size):
    batch = entries[batch_start:batch_start + batch_size]
    
    with mp.Pool(n_cores) as pool:
        results = pool.map(process_image, batch)
    
    # Absorb into landscapes (sequential — Welford needs order)
    for char_list in results:
        for label, fv in char_list:
            if label not in landscapes:
                landscapes[label] = Landscape(label)
            landscapes[label].absorb(fv)
            total_chars += 1
    
    elapsed = time.time() - start
    rate = total_chars / max(elapsed, 1)
    batch_num = batch_start // batch_size + 1
    total_batches = (len(entries) + batch_size - 1) // batch_size
    print(f'  Batch {batch_num}/{total_batches}: {total_chars} chars, {rate:.0f}/sec, {elapsed:.0f}s')

elapsed = time.time() - start
print(f'\nTraining complete: {total_chars} characters in {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'Landscapes: {len(landscapes)}')
for label, land in sorted(landscapes.items()):
    print(f'  {label}: {land.n} observations')

In [ ]:
# Step 7: Save model

model = {
    'version': '2.0.0',
    'total_predictions': 0,
    'total_correct': 0,
    'landscapes': {k: v.to_dict() for k, v in landscapes.items()},
}

model_path = 'trained_model.json'
with open(model_path, 'w') as f:
    json.dump(model, f)

size_mb = os.path.getsize(model_path) / 1024 / 1024
print(f'Model saved: {model_path} ({size_mb:.1f} MB)')

In [ ]:
# Step 8: Quick validation test

val_list_path = os.path.join(TRAINING_DIR, 'val_list.txt')
val_entries = []
if os.path.exists(val_list_path):
    with open(val_list_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or '\t' not in line: continue
            img_path, label = line.split('\t', 1)
            full_path = os.path.join(TRAINING_DIR, img_path)
            if os.path.exists(full_path):
                val_entries.append((full_path, label))

print(f'Validating on {min(len(val_entries), 500)} images...')

correct = 0
tested = 0

for img_path, label in val_entries[:500]:
    try:
        img = Image.open(img_path).convert('L')
        arr = np.array(img)
        img.close()
        segments = segment_characters(arr, min_char_width=2)
        if not segments: continue
        clean_chars = [c.lower() for c in label if c.lower() in ALLOWED_CHARS]
        if not clean_chars: continue
        n = min(len(segments), len(clean_chars))
        if abs(len(segments) - len(clean_chars)) / max(len(clean_chars), 1) > 0.2: continue
        for i in range(n):
            binary = fast_binarize(segments[i])
            region = normalize_region(extract_region(binary))
            fv = extract_features(region)
            # Predict
            scores = [(lbl, land.fit(fv)) for lbl, land in landscapes.items()]
            scores.sort(key=lambda x: x[1], reverse=True)
            predicted = scores[0][0] if scores else '?'
            tested += 1
            if predicted == clean_chars[i]:
                correct += 1
    except:
        continue

print(f'Validation accuracy: {correct}/{tested} ({100*correct/max(tested,1):.1f}%)')

In [ ]:
# Step 9: Download the trained model
from google.colab import files
files.download('trained_model.json')
print('Download started — copy to shifu-ocr/shifu_ocr/trained_model.json')